In [0]:
%sql
DROP TABLE la_lakehouse.silver.la_parcels

In [0]:
%pip install shapely
%restart_python

In [0]:
from pyspark.sql.functions import col, trim 
from pyspark.sql import functions as F
from pyspark.sql.types import BinaryType
from shapely.wkt import loads

#Reading From Bronze

In [0]:
df = spark.table("la_lakehouse.bronze.la_parcels")

In [0]:
# TODO
# convert all columns to their correct data type
# strip the whitespace for pin 
# normalize the columns

df.display()

#Coverting the Data Types

In [0]:
@F.udf(returnType=BinaryType())
def wkt_to_wkb(wkt_str):
    if wkt_str is None:
        return None
    try: 
        return loads(wkt_str).wkb
    except Exception as e:
        print(f"Error converting WKT to WKB: {e}")

columns_to_cast = {
    "assetid": "int",
    "id": "int",
    "shape_leng": "float",
    "shape_area": "float",
    "_updated_at": "timestamp"
}

df = df.withColumn("the_geom", wkt_to_wkb(col("the_geom")))
df = df.select([
    F.regexp_replace(col(c), ",", "").cast("float").cast("int").alias(c) if c in ["assetid", "id"] else
    F.regexp_replace(col(c), ",", "").cast(columns_to_cast[c]).alias(c) if c in columns_to_cast else col(c) 
    for c in df.columns
])


In [0]:
df.printSchema()

In [0]:
df.display()

#Stripping The Whitespace

In [0]:
df = df.withColumn("pin",F.regexp_replace(col("pin"),r"\s+",""))

In [0]:
df.display()

#Writing to the Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("la_lakehouse.silver.la_parcels")
)

#Testing

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.silver.la_parcels

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.silver.la_parcels WHERE the_geom IS NULL

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.silver.la_parcels WHERE assetid IS NULL

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.silver.la_parcels WHERE shape_area IS NULL

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.silver.la_parcels WHERE pin IS NULL

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.silver.la_parcels WHERE shape_leng IS NULL